#### Building A Chatbot

In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for.

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from google import genai
gemini_api_key=os.getenv('GEMINI_API_KEY')
groq_api_key=os.getenv('GROQ_API_KEY')

# groq_api_key

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

c:\Users\kalya\Projects\Py_Prj_D300526\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000189C4C03770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000189C4DF4590>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    HumanMessage(content="Hello, My name is Kris and I'm a Chief AI Engineer")
]

res=model.invoke(messages)
res

AIMessage(content="Hello Kris, nice to meet you. It's great to connect with a Chief AI Engineer like yourself. What brings you here today? Are you looking to discuss any AI-related projects, share your expertise, or explore new ideas? I'm all ears.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 48, 'total_tokens': 100, 'completion_time': 0.16098471, 'completion_tokens_details': None, 'prompt_time': 0.002234973, 'prompt_tokens_details': None, 'queue_time': 0.046762227, 'total_time': 0.163219683}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec1e1-d543-7373-9f51-f2d2d2a0316b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 52, 'total_tokens': 100})

In [6]:
from langchain_core.messages import AIMessage

model.invoke([
    HumanMessage(content="Hello, My name is Kris and I'm a Chief AI Engineer"),
    AIMessage(content="Hello Kris, nice to meet you. As a Chief AI Engineer, you must be working on some exciting and innovative projects. What type of AI applications are you currently focused on, and what industries or domains are you applying them to? I'm here to help and learn from your expertise, so feel free to share your experiences and insights.\n"),
    HumanMessage(content="Hey, What's my name and what do I do?")
])

AIMessage(content="Your name is Kris, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 138, 'total_tokens': 152, 'completion_time': 0.029817097, 'completion_tokens_details': None, 'prompt_time': 0.031037778, 'prompt_tokens_details': None, 'queue_time': 0.163223762, 'total_time': 0.060854875}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec1e1-dfc7-7f60-a5e9-83bd76ae1a0f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 14, 'total_tokens': 152})

### Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

# to differentiate btw sessions, we can create a function to do this thing!!!
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

C:\Users\kalya\AppData\Local\Temp\ipykernel_16636\192144794.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\kalya\Projects\Py_Prj_D300526\venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
#created a configuration
config={"configurable":{"session_id":"chat1"}}

In [9]:
#use this sessionid and chat1 in our llm model
response=with_message_history.invoke(
    [HumanMessage(content="Hello, My name is Kris and I'm a Chief AI Engineer")],
    config=config
)
response

AIMessage(content="Hello Kris, nice to meet you. It's great to connect with a Chief AI Engineer like yourself. What brings you here today? Are you working on any exciting AI projects that you'd like to discuss or are you looking for assistance with a specific challenge? I'm all ears.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 48, 'total_tokens': 106, 'completion_time': 0.164831278, 'completion_tokens_details': None, 'prompt_time': 0.006952045, 'prompt_tokens_details': None, 'queue_time': 0.046529955, 'total_time': 0.171783323}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec1e1-f909-7802-8623-cc694ff82444-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 58, 'total_tokens': 106})

In [10]:
response.content

"Hello Kris, nice to meet you. It's great to connect with a Chief AI Engineer like yourself. What brings you here today? Are you working on any exciting AI projects that you'd like to discuss or are you looking for assistance with a specific challenge? I'm all ears."

In [11]:
#using the message history, so that it is remembering who am i? and telling that
with_message_history.invoke(
    [HumanMessage(content="Who am I?")],
    config=config
)

AIMessage(content="You are Kris, and you've introduced yourself as a Chief AI Engineer. That suggests you're a high-level professional with expertise in artificial intelligence, likely leading a team or overseeing the development of AI-related projects and technologies. Is that a correct assumption?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 119, 'total_tokens': 170, 'completion_time': 0.214267499, 'completion_tokens_details': None, 'prompt_time': 0.005114359, 'prompt_tokens_details': None, 'queue_time': 0.161677661, 'total_time': 0.219381858}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec1e2-0aea-7812-b1c2-504c1956ad2c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 51, 'total_tokens': 170})

In [12]:
# so, let's change the config ---> session_id
#since i changed the config, and session id from chat1 to chat2, it is not able to remember and fetch history!

config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)

response.content

"I don't know your name. I'm a large language model, I don't have any information about you, including your name. I'm here to help answer your questions and provide information, but I don't have any personal knowledge about you. Would you like to introduce yourself?"

In [13]:
#now, let's give another name here for chat2 config, let's see!!!
with_message_history.invoke(
    [HumanMessage(content="I am Sam!!!")],
    config=config1
)

AIMessage(content="Hello Sam. It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 111, 'total_tokens': 137, 'completion_time': 0.052957018, 'completion_tokens_details': None, 'prompt_time': 0.006493928, 'prompt_tokens_details': None, 'queue_time': 0.046781562, 'total_time': 0.059450946}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec1e2-192f-7ec1-9ec8-ac75e2d890cd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 26, 'total_tokens': 137})

In [14]:
#now, let's try asking my name
response=with_message_history.invoke(
    [HumanMessage(content="Say my name!!!")],
    config=config1
)

response.content

'SAM!!!'

#### Prompt Templates
Prompt templates help to run raw user info into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to LLM. Let's now make that a bit more complicated. First, let's add in a system with some custom instructioons(but still taking message as input). Next, we'll add in more input besides just the messages.

In [15]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model
chain

ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.mes

In [16]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")]})

AIMessage(content="Hello Krish, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 56, 'total_tokens': 82, 'completion_time': 0.067194245, 'completion_tokens_details': None, 'prompt_time': 0.002964746, 'prompt_tokens_details': None, 'queue_time': 0.046352213, 'total_time': 0.070158991}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec1e2-31d3-7532-8ae3-03b6e0579047-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 26, 'total_tokens': 82})

In [17]:
with_msg_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Users\kalya\Projects\Py_Prj_D300526\venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [18]:
config2={"configurable":{"session_id":"chat3"}}
response=with_msg_history.invoke(
    [HumanMessage(content="Hi, My name is Ram")],
    config=config2
)

response.content

"Hello Ram, it's nice to meet you. Is there something I can help you with or would you like to chat?"

In [19]:
#Add more complexity
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the question to the best of your ability in {language}."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model
chain

ChatPromptTemplate(input_variables=['language', 'messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langch

In [20]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Ram")],"language":"Telugu"})
response

AIMessage(content='హాయ్ రామ్, నమస్తే. నేను మీకు ఎలా సహాయం చేయగలను?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 60, 'total_tokens': 138, 'completion_time': 0.225751221, 'completion_tokens_details': None, 'prompt_time': 0.002726999, 'prompt_tokens_details': None, 'queue_time': 0.340527781, 'total_time': 0.22847822}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec1e2-5897-75d2-ba65-85c884b250b5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 78, 'total_tokens': 138})

In [21]:
response.content

'హాయ్ రామ్, నమస్తే. నేను మీకు ఎలా సహాయం చేయగలను?'

Let's now wrap this more complicated chain in a Message History Class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history

In [22]:
with_message_history2=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [23]:
config3={"configurable":{"session_id":"chat4"}}

response=with_message_history2.invoke(
    {"messages":[HumanMessage(content="Hi, I am Venky")],"language":"Telugu"},
    config=config3
)

response.content

'హాయ్ వెంకీ, నేను మీకు ఎలా సహాయం చేయగలను?'

In [24]:
response=with_message_history2.invoke(
    {"messages":[HumanMessage(content="Hi, What's my name?")],"language":"Telugu"},
    config=config3
)

response.content

'నీ పేరు వెంకీ. నీవు ముందు చెప్పావు కదా!'

### Manage the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in

'trim_messages' helper to reduce how many msgs we're sending to the model, The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we wnat to always keep the system msg, and whether to allow partial msgs

In [29]:
from langchain_core.messages import SystemMessage,trim_messages

trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt
    |model   
)

response=chain.invoke(
    {
        "messages":messages+[HumanMessage(content="What icecream do i like?")],
        "language":"Telugu"
    }
)

#since, we trimmed the msgs, the context got missed, that's why it is not able to give answer for this question for below!!
response.content

'నేను మీకు ఇష్టమైన ఐస్\u200cక్రీం రుచిని గుర్తు చేసుకోలేను, ఎందుకంటే మీరు నాకు దాని గురించి చెప్పలేదు. మీరు నాకు చెప్పినట్లయితే నేను గుర్తు చేసుకోగలను!'

In [31]:
#now let's try for another simple one

response=chain.invoke(
    {
        "messages":messages+[HumanMessage(content="what's 2+2?")],
        "language":"Telugu"
    }
)

response.content

'అది 4. నాకు సహాయం చేయడంలో ఆనందంగా ఉంది.'

In [32]:
#now, let's wrap this in Message History!!!
with_msg_historie=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

config4={"configurable":{"session_id":"chat5"}}

c:\Users\kalya\Projects\Py_Prj_D300526\venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [34]:
#since we added the msgs in history, let's see by invoking using above historie!!!
response=with_msg_historie.invoke(
    {
        "messages":messages+[HumanMessage(content="what's fav ice?")],
        "language":"Telugu"
    },
    config=config4
)

response.content

'వానిలా ఐస్ క్రీం మంచిది కదా!'